# KorQuAD QLoRA 파인튜닝 — Kaggle 무인 실행판 (Qwen2.5-1.5B-Instruct)

qlora_finetune.ipynb(Colab용)와 동일한 로직을 Kaggle Notebooks의 백그라운드 커밋
실행에 맞게 재구성했다. Drive 마운트·백업/복원·재개·스모크 테스트 등 Colab 전용
셀은 제외하고, 위에서 아래로 사람 개입 없이 완주하도록 구성했다.

**실행 전 반드시 설정할 것 (3가지):**
1. Settings → Accelerator를 **GPU T4 x2**로 설정한다.
2. Settings → Internet을 **ON**으로 설정한다 (git clone·pip install에 필요).
3. 위 두 설정을 마친 뒤 Save Version → **Save & Run All (Commit)**을 선택한다.
   이 방식으로 실행하면 백그라운드에서 무인으로 전체 셀이 순차 실행된다.

전체 예상 소요 시간은 약 4시간이며, 전 과정이 무인으로 진행된다.

In [ ]:
!nvidia-smi

In [ ]:
%cd /kaggle/working
!rm -rf korquad-qlora-lab
!git clone https://github.com/calintzy/korquad-qlora-lab.git
%cd korquad-qlora-lab

In [ ]:
!pip install -q -r requirements.txt

채점기 자가검증 (ISC-1.2/1.3) — 실패해도 다음 셀로 진행되지만 이 출력이 기록으로 남는 것이 목적이다.

In [ ]:
!python tests/test_scorer.py

고정 시드 42라 Colab 측정치(EM 38.6 / F1 62.77)와 동일 문항·동일 결과가 재현되어야 함 (재현 자체가 검증).

In [ ]:
!CUDA_VISIBLE_DEVICES=0 python src/eval_run.py \
  --output results/results_before.json \
  --subset-n 1000 \
  --predictions-out results/predictions_before.json

In [ ]:
!CUDA_VISIBLE_DEVICES=0 python src/train.py \
  --output-dir /kaggle/working/adapter \
  --epochs 1 \
  --max-steps 1000

In [ ]:
!CUDA_VISIBLE_DEVICES=0 python src/eval_run.py \
  --adapter /kaggle/working/adapter \
  --output results/results_after.json \
  --predictions-out results/predictions_after.json

In [ ]:
import json

with open("results/results_before.json") as f:
    before = json.load(f)
with open("results/results_after.json") as f:
    after = json.load(f)

print(f"{'구분':<10}{'EM':>10}{'F1':>10}{'n':>8}")
print(f"{'before':<10}{before['em']:>10.2f}{before['f1']:>10.2f}{before['n']:>8}")
print(f"{'after':<10}{after['em']:>10.2f}{after['f1']:>10.2f}{after['n']:>8}")
print(f"{'delta':<10}{after['em']-before['em']:>10.2f}{after['f1']-before['f1']:>10.2f}")

In [ ]:
import shutil, os
os.makedirs('/kaggle/working/results_final', exist_ok=True)
shutil.copy('results/results_before.json', '/kaggle/working/results_final/')
shutil.copy('results/results_after.json', '/kaggle/working/results_final/')

In [ ]:
!ls -la /kaggle/working/results_final /kaggle/working/adapter

실행이 모두 끝나면 노트북 페이지의 Output 탭에서 `/kaggle/working/results_final`과 `/kaggle/working/adapter`를 다운로드한다.